# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets and their field @id's
record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record sets.")
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    print(f"  name: {rs.get('name', '<no name>')}")
    fields = rs.get('fields', [])
    if fields:
        print("  Fields:")
        for field in fields:
            print(f"    - @id: {field['@id']} | name: {field.get('name', '<no name>')} | dataType: {field.get('dataType', '<unknown>')}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Example: Let's extract data from all available record sets
import pprint

dataframes = {}
# Store mapping of names for convenience
recordset_names = {}
for rs in record_sets:
    rec_id = rs['@id']
    recordset_names[rec_id] = rs.get('name', rec_id)
    print(f"Loading RecordSet: {rec_id} ({rs.get('name', '')})")
    try:
        records = list(dataset.records(record_set=rec_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rec_id] = df
            print(f"  Columns: {df.columns.tolist()}")
            print(df.head())
        else:
            print("  No records found.")
    except Exception as e:
        print(f"  Could not load records: {e}")
    print("-"*64)

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Choose a record set and field for EDA
# (Update these @id values if the schema changes)
# Here, we use the first record set found with a numeric field.
chosen_record_set = None
chosen_numeric_field = None
chosen_group_field = None

# Identify a suitable DataFrame and numeric field
for rs in record_sets:
    rec_id = rs['@id']
    fields = rs.get('fields', [])
    df = dataframes.get(rec_id, pd.DataFrame())
    # Try to find a float/integer field
    for field in fields:
        f_id = field['@id']
        if field.get('dataType') in ('schema:Float', 'schema:Integer', 'Float', 'Integer'):
            if f_id in df.columns:
                chosen_record_set = rec_id
                chosen_numeric_field = f_id
                # Try a group field different from numeric
                for g_field in fields:
                    if g_field['@id'] != f_id and g_field['@id'] in df.columns and g_field.get('dataType') != 'schema:Float':
                        chosen_group_field = g_field['@id']
                        break
                break
    if chosen_record_set:
        break

if (chosen_record_set is None) or (chosen_numeric_field is None):
    print("No suitable numeric records found for EDA.")
else:
    df = dataframes[chosen_record_set]
    print(f"Selected RecordSet: '{chosen_record_set}' ({recordset_names[chosen_record_set]})")
    print(f"Selected Numeric Field @id: '{chosen_numeric_field}'")
    if chosen_group_field:
        print(f"Group by Field @id: '{chosen_group_field}'")
    else:
        print("No grouping field identified.")

    # Drop rows with missing values in the numeric field
    numeric_col = chosen_numeric_field
    filter_threshold = None
    # Try to get threshold from quantile
    if pd.api.types.is_numeric_dtype(df[numeric_col]):
        filter_threshold = df[numeric_col].quantile(0.25)  # e.g. first quartile as cutoff
    else:
        # Convert values if initially loaded as str
        df[numeric_col] = pd.to_numeric(df[numeric_col], errors='coerce')
        filter_threshold = df[numeric_col].quantile(0.25)
    print(f"Using threshold: {filter_threshold}")

    filtered_df = df[df[numeric_col] > filter_threshold].copy()
    print(f"Filtered {len(filtered_df)} records with {numeric_col} > {filter_threshold}.")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_col}_normalized"] = (filtered_df[numeric_col] - filtered_df[numeric_col].mean()) / filtered_df[numeric_col].std()
    print(filtered_df[[numeric_col, f"{numeric_col}_normalized"]].head())

    # Group by a field and aggregate
    if chosen_group_field and chosen_group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(chosen_group_field)[numeric_col].mean().reset_index().sort_values(by=numeric_col, ascending=False)
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field (before/after normalization)
if (chosen_record_set is not None) and len(filtered_df) > 0:
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_col], kde=True, color='skyblue')
    plt.title(f"Distribution of field: {numeric_col}")
    plt.xlabel(numeric_col)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

    # If grouping field exists, show average value per group
    if chosen_group_field and chosen_group_field in filtered_df.columns:
        plt.figure(figsize=(10, 4))
        order = filtered_df.groupby(chosen_group_field)[numeric_col].mean().sort_values(ascending=False).index
        sns.barplot(data=filtered_df, x=chosen_group_field, y=numeric_col, order=order, ci=None)
        plt.xticks(rotation=60)
        plt.title(f"Mean of {numeric_col} by group: {chosen_group_field}")
        plt.tight_layout()
        plt.show()
else:
    print("No suitable data for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we successfully loaded and explored the FAIR^2 dataset:

- Utilized the Croissant schema and `mlcroissant` to review the complete structure, including all available record sets, fields, and their `@id` identifiers.
- Loaded data for all record sets and inspected their columns and contents.
- Chose representative numeric and grouping fields (referenced by their `@id`), performed basic filtering and normalization, and grouped values for summary statistics.
- Visualized the filtered and normalized data distributions and mean values across groups.

This workflow enables programmatic and reproducible FAIR data exploration, with precise referencing via `@id` for downstream data science tasks.